# 02 Predictions And External Evaluation

Hand-maintained revision runbook. Do not regenerate from `scripts/revision/create_revision_notebooks.py` unless that generator is first updated to the fixed-epoch final_ema/model_runs protocol.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
chapman_candidates = [
    DRIVE_ROOT / 'WFDB-ChapmanShaoxing.zip',
    DRIVE_ROOT / 'WFDB_ChapmanShaoxing.zip',
    DRIVE_ROOT / 'chapman.zip',
    DRIVE_ROOT / 'archive.zip',
]
chapman_zip = next((path for path in chapman_candidates if path.is_file()), None)
if chapman_zip is None:
    visible_zips = sorted(path.name for path in DRIVE_ROOT.glob('*.zip')) if DRIVE_ROOT.exists() else []
    raise FileNotFoundError(
        'Chapman dataset ZIP was not found. Checked: '
        + ', '.join(str(path) for path in chapman_candidates)
        + f'. ZIP files currently visible under {DRIVE_ROOT}: {visible_zips}'
    )
if chapman_zip.stat().st_size == 0 or not zipfile.is_zipfile(chapman_zip):
    raise ValueError(f'Chapman dataset is not a valid non-empty ZIP: {chapman_zip}')
model_pointer = DRIVE_ROOT / 'model_runs' / 'current_final_ema_model_dir.txt'
model_dir_env = os.environ.get('ECG_RAMBA_MODEL_DIR')
if model_dir_env:
    model_dir = Path(model_dir_env).expanduser()
elif model_pointer.is_file():
    model_dir = Path(model_pointer.read_text(encoding='utf-8').strip()).expanduser()
else:
    model_dir = DRIVE_ROOT / 'model'
if not model_dir.is_dir():
    raise FileNotFoundError(f'Model directory not found: {model_dir}. Run notebook 02a first or set ECG_RAMBA_MODEL_DIR.')
os.environ['ECG_RAMBA_CHAPMAN_ZIP'] = str(chapman_zip)
os.environ['ECG_RAMBA_MODEL_DIR'] = str(model_dir)
print('Model dir:', model_dir)
print('Model pointer:', model_pointer, 'exists=', model_pointer.is_file())
os.environ['ECG_RAMBA_LOCAL_ROOT'] = str(LOCAL_RUNTIME_ROOT)
os.environ['ECG_RAMBA_EXTRACT_DIR'] = str(LOCAL_RUNTIME_ROOT / 'chapman')
os.environ['ECG_RAMBA_USE_CLEAN_CACHE'] = '1'
os.environ['ECG_RAMBA_SAVE_CLEAN_CACHE'] = '0'

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('chapman_zip:', chapman_zip, '| size=', chapman_zip.stat().st_size)
print('model_dir  :', model_dir)
print('branch    :', BRANCH)

from configs.config import PATHS

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache_pattern': 'rocket_raw_N44186_C12_L5000_K10000_S42*.npz',
    'hrv36_cache_pattern': 'hrv36_N44186_C12_L5000*.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_pca_models',
    'fold_hydra_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, target in cache_status.items():
    if isinstance(target, str):
        matches = sorted(DRIVE_ROOT.glob(target))
        print(f'  {name}: count={len(matches)} pattern={target}')
        for match in matches[:5]:
            print(f'    - {match.name} size={match.stat().st_size}')
    else:
        path = target
        if path.is_dir():
            npz_count = len(list(path.glob('*.npz')))
            joblib_count = len(list(path.glob('*.joblib')))
            print(f'  {name}: exists=True npz_count={npz_count} joblib_count={joblib_count} path={path}')
        else:
            print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

# Expected feature-cache names are fingerprinted from the cleaned cache subject order.
# If these files exist, train/evaluation will load them instead of regenerating MiniRocket/HRV.
def inspect_expected_feature_caches():
    import numpy as np
    from src.provenance import record_order_fingerprint

    clean_cache_path = Path(PATHS['data_cache'])
    if not clean_cache_path.is_file():
        print('Feature cache preflight: clean cache is missing, so exact MiniRocket/HRV fingerprint cannot be derived yet.')
        return

    with np.load(clean_cache_path, allow_pickle=True) as payload:
        subjects = payload['subjects']
        stored_fingerprint = (
            str(payload['record_order_fingerprint'].item())
            if 'record_order_fingerprint' in payload.files
            else None
        )
    computed_fingerprint = record_order_fingerprint(subjects)
    if stored_fingerprint and stored_fingerprint != computed_fingerprint:
        raise RuntimeError(
            'Clean cache fingerprint mismatch: '
            f'stored={stored_fingerprint} computed={computed_fingerprint}'
        )
    fingerprint = stored_fingerprint or computed_fingerprint
    n_records = len(subjects)
    cache_dir = Path(PATHS['cache_dir'])
    expected_feature_caches = {
        'expected RAW MiniRocket cache': cache_dir / f'rocket_raw_N{n_records}_C12_L5000_K10000_S42_R{fingerprint}.npz',
        'expected HRV36 cache': cache_dir / f'hrv36_N{n_records}_C12_L5000_R{fingerprint}.npz',
    }
    print('Feature cache fingerprint:', fingerprint)
    all_present = True
    for name, path in expected_feature_caches.items():
        exists = path.is_file()
        all_present = all_present and exists
        size = path.stat().st_size if exists else None
        size_gib = f'{size / (1024 ** 3):.2f} GiB' if size is not None else '-'
        print(f'{name:32s}: exists={exists} size={size_gib} path={path}')
    if all_present:
        print('Feature cache preflight: MiniRocket and HRV36 caches are present and should be reused.')
    else:
        print('Feature cache preflight: at least one feature cache is missing; that feature will be regenerated once and then cached.')

inspect_expected_feature_caches()

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Install Base Dependencies

This notebook can run directly after opening in Colab. These packages cover data loading, feature extraction, metrics, and artifact checks.

In [ ]:
INSTALL_BASE_DEPS = True
RESTART_RUNTIME_AFTER_BASE_DEP_CHANGE = True
REPAIR_BROKEN_NUMERIC_STACK = True

if INSTALL_BASE_DEPS:
    import importlib
    import importlib.metadata as metadata
    import json
    import os
    import subprocess
    import sys
    import time

    numeric_packages = [
        "numpy>=2.0,<2.1",
        "scipy>=1.14.1,<2.0",
        "scikit-learn>=1.2,<1.9",
    ]
    support_packages = [
        "pandas==2.2.2",
        "requests==2.32.4",
        "pillow>=8,<12",
        "threadpoolctl",
        "tqdm",
        "wfdb==4.1.2",
        "joblib",
        "matplotlib",
        "seaborn",
        "packaging",
        "neurokit2",
        "iterative-stratification",
        "thop",
        "einops",
        "ninja",
    ]
    numeric_dists = {
        "numpy": "numpy",
        "scipy": "scipy",
        "scikit-learn": "scikit-learn",
    }
    runtime_dir = DRIVE_ROOT / "colab_runtime"
    runtime_dir.mkdir(parents=True, exist_ok=True)

    def dist_version(dist_name):
        try:
            return metadata.version(dist_name)
        except metadata.PackageNotFoundError:
            return None

    def numeric_versions():
        return {name: dist_version(dist) for name, dist in numeric_dists.items()}

    def pip_install(extra_args, packages):
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                *extra_args,
                *packages,
            ],
            check=True,
        )

    def restart_runtime(reason):
        print("")
        print("=" * 80)
        print("Intentional Colab runtime restart")
        print("=" * 80)
        print(reason)
        print("After Colab reconnects, run this notebook again from the first cell.")
        print("Base dependency state is stored under:", runtime_dir)
        print("=" * 80)
        sys.stdout.flush()
        time.sleep(8)
        try:
            from google.colab import runtime
            runtime.restart_runtime()
        except Exception:
            os.kill(os.getpid(), 9)

    def sanity_import_numeric_stack():
        import numpy as np
        from numpy._core import strings as _np_strings  # catches mixed NumPy installs on Colab
        import scipy
        import sklearn
        import wfdb
        return {
            "numpy": np.__version__,
            "scipy": scipy.__version__,
            "sklearn": sklearn.__version__,
            "wfdb": wfdb.__version__,
        }

    print("Python:", sys.version)
    print("Installing/updating ECG-RAMBA runtime dependencies without downgrading Colab numpy/scipy.")
    before_versions = numeric_versions()
    pip_install(["--upgrade", "--upgrade-strategy", "only-if-needed"], numeric_packages + support_packages)
    after_versions = numeric_versions()
    changed_versions = {
        name: {"before": before_versions.get(name), "after": after_versions.get(name)}
        for name in numeric_dists
        if before_versions.get(name) != after_versions.get(name)
    }
    if changed_versions and RESTART_RUNTIME_AFTER_BASE_DEP_CHANGE:
        manifest = {
            "reason": "Base numeric dependencies changed; restart required before importing sklearn/NumPy extensions.",
            "changed_versions": changed_versions,
            "python": sys.version,
        }
        manifest_path = runtime_dir / f"base_deps_restart_py{sys.version_info.major}{sys.version_info.minor}.json"
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        restart_runtime("Base numeric dependencies changed: " + json.dumps(changed_versions))

    try:
        versions = sanity_import_numeric_stack()
    except Exception as exc:
        print("Numeric stack import sanity check failed:", repr(exc))
        if REPAIR_BROKEN_NUMERIC_STACK:
            print("Force-reinstalling NumPy/SciPy/scikit-learn once, then restarting runtime.")
            pip_install(["--force-reinstall", "--no-cache-dir"], numeric_packages)
            repair_manifest = {
                "reason": "Numeric stack import failed after pip install; repaired with force-reinstall.",
                "error": repr(exc),
                "versions_before_repair": after_versions,
                "versions_after_repair": numeric_versions(),
                "python": sys.version,
            }
            repair_path = runtime_dir / f"numeric_stack_repair_py{sys.version_info.major}{sys.version_info.minor}.json"
            repair_path.write_text(json.dumps(repair_manifest, indent=2), encoding="utf-8")
            restart_runtime("NumPy/SciPy/scikit-learn were force-reinstalled after an import failure.")
        raise

    for name, version in versions.items():
        print(f"{name:8s}: {version}")

    check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if check.returncode:
        print("pip check reported remaining non-ECG Colab package conflicts:")
        print(check.stdout)
        print("Continue only if the ECG-RAMBA imports above succeeded; unused Colab packages can still report conflicts.")
    else:
        print("pip check: OK")
else:
    print('Skipping base dependency install.')


## Install Model Dependencies

In [ ]:
INSTALL_MODEL_DEPS = True
DOWNLOAD_MAMBA_WHEELS_IF_MISSING = True
AUTO_PIN_TORCH_FOR_MAMBA = True
RESTART_RUNTIME_AFTER_TORCH_PIN = True
UNINSTALL_TORCH_COMPANION_PACKAGES = True

if INSTALL_MODEL_DEPS:
    import hashlib
    import json
    import os
    import re
    import subprocess
    import sys
    import time
    import urllib.request

    import torch

    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_version = torch.version.cuda or ""
    cuda_tag = f"cu{cuda_version.split('.')[0]}" if cuda_version else None
    abi_tag = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"

    wheel_cache_dir = DRIVE_ROOT / f"mamba_wheels_py{sys.version_info.major}{sys.version_info.minor}"
    legacy_dirs = [
        Path("/content/drive/MyDrive") / wheel_cache_dir.name,
        Path("/content/drive/MyDrive/mamba_wheels_py312"),
        DRIVE_ROOT / "mamba_wheels_py312",
    ]
    explicit_wheel_dirs = []
    for p in [wheel_cache_dir, *legacy_dirs]:
        if p not in explicit_wheel_dirs:
            explicit_wheel_dirs.append(p)

    print("Mamba wheel environment")
    print("  Python tag :", py_tag)
    print("  torch      :", torch.__version__)
    print("  CUDA       :", cuda_version or "CPU")
    print("  CUDA tag   :", cuda_tag or "none")
    print("  CXX11 ABI  :", abi_tag)
    print("  cache dir  :", wheel_cache_dir)

    if cuda_tag is None:
        raise RuntimeError("CUDA-enabled PyTorch is required. In Colab, select a GPU runtime before running this notebook.")

    release_cache = {}
    torch_companion_packages = ["torchvision", "torchaudio", "torchtext"]

    def restart_runtime_after_pin() -> None:
        print("")
        print("=" * 80)
        print("Intentional Colab runtime restart")
        print("=" * 80)
        print("Torch was changed to a Mamba-compatible version.")
        print("After Colab reconnects, run this notebook again from the first cell.")
        print("The downloaded/pinned artifacts are stored on Drive and will be reused.")
        print("=" * 80)
        sys.stdout.flush()
        time.sleep(8)
        try:
            from google.colab import runtime
            runtime.restart_runtime()
        except Exception:
            os.kill(os.getpid(), 9)

    def sha256_file(path: Path) -> str:
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    def wheel_role(path_or_name) -> str | None:
        name = Path(path_or_name).name
        if name.startswith("causal_conv1d-"):
            return "causal"
        if name.startswith("mamba_ssm-"):
            return "mamba"
        return None

    def compatible_wheel(path_or_name, role: str, torch_minor: str | None = None) -> bool:
        name = Path(path_or_name).name
        if wheel_role(name) != role:
            return False
        torch_minor = torch_minor or torch_major_minor
        required = [
            f"{py_tag}-{py_tag}",
            "linux_x86_64.whl",
            f"torch{torch_minor}",
            f"cxx11abi{abi_tag}",
        ]
        if cuda_tag:
            required.append(cuda_tag)
        return all(token in name for token in required)

    def list_wheels():
        wheels = []
        for wheel_dir in explicit_wheel_dirs:
            if wheel_dir.exists():
                wheels.extend(sorted(wheel_dir.glob("*.whl")))
        return sorted(set(wheels))

    def select_cached_wheels():
        wheels = list_wheels()
        if wheels:
            print(f"Found {len(wheels)} wheel file(s) in configured Drive cache folders.")
            for p in wheels[:60]:
                flag = "compatible" if compatible_wheel(p, wheel_role(p) or "") else "present"
                print(f"  - {p.name} [{flag}]")
            if len(wheels) > 60:
                print(f"  ... {len(wheels) - 60} more wheel(s)")
        causal = [p for p in wheels if compatible_wheel(p, "causal")]
        mamba = [p for p in wheels if compatible_wheel(p, "mamba")]
        return (causal[0] if causal else None), (mamba[0] if mamba else None)

    def github_latest_release(repo: str) -> dict:
        if repo in release_cache:
            return release_cache[repo]
        url = f"https://api.github.com/repos/{repo}/releases/latest"
        req = urllib.request.Request(url, headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            release_cache[repo] = json.loads(resp.read().decode("utf-8"))
        return release_cache[repo]

    def asset_torch_minor(asset_name: str, role: str) -> str | None:
        if wheel_role(asset_name) != role:
            return None
        required = [
            f"{py_tag}-{py_tag}",
            "linux_x86_64.whl",
            f"cxx11abi{abi_tag}",
        ]
        if cuda_tag:
            required.append(cuda_tag)
        if not all(token in asset_name for token in required):
            return None
        m = re.search(r"torch([0-9]+\.[0-9]+)", asset_name)
        return m.group(1) if m else None

    def release_supported_torch_minors(repo: str, role: str) -> set[str]:
        rel = github_latest_release(repo)
        minors = {
            minor for asset in rel.get("assets", [])
            if (minor := asset_torch_minor(asset["name"], role)) is not None
        }
        return minors

    def torch_minor_key(minor: str) -> tuple[int, int]:
        major, minor_part = minor.split(".", 1)
        return int(major), int(minor_part)

    def choose_supported_torch_minor() -> tuple[str | None, list[str]]:
        causal_minors = release_supported_torch_minors("Dao-AILab/causal-conv1d", "causal")
        mamba_minors = release_supported_torch_minors("state-spaces/mamba", "mamba")
        common = sorted(causal_minors & mamba_minors, key=torch_minor_key)
        common_stable = [m for m in common if m.startswith("2.")]
        if torch_major_minor in common:
            return torch_major_minor, common

        requested = os.environ.get("ECG_RAMBA_TORCH_MINOR")
        if requested:
            if requested not in common:
                raise RuntimeError(f"ECG_RAMBA_TORCH_MINOR={requested} is not supported by both Mamba wheel releases: {common}")
            return requested, common

        lower_or_equal = [
            m for m in common_stable
            if torch_minor_key(m) <= torch_minor_key(torch_major_minor)
        ]
        if lower_or_equal:
            return lower_or_equal[-1], common
        if common_stable:
            return common_stable[-1], common
        return None, common

    def version_tuple(version: str) -> tuple[int, int, int]:
        parts = version.split("+", 1)[0].split(".")
        nums = [int(p) if p.isdigit() else 0 for p in parts[:3]]
        while len(nums) < 3:
            nums.append(0)
        return tuple(nums)

    def latest_pypi_torch_version(target_minor: str) -> str:
        url = "https://pypi.org/pypi/torch/json"
        req = urllib.request.Request(url, headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        versions = [
            version for version, files in payload.get("releases", {}).items()
            if version.startswith(target_minor + ".")
            and re.match(r"^[0-9]+\.[0-9]+\.[0-9]+$", version)
            and files
        ]
        if not versions:
            raise RuntimeError(f"No stable torch release found on PyPI for torch{target_minor}.")
        return sorted(versions, key=version_tuple)[-1]

    def pin_torch_if_needed() -> None:
        target_minor, supported = choose_supported_torch_minor()
        print("Mamba-supported torch minors:", supported)
        if target_minor == torch_major_minor:
            return
        if target_minor is None:
            raise RuntimeError(
                f"No common Mamba wheel support found for {py_tag}, {cuda_tag}, cxx11abi{abi_tag}. "
                "Use a different Colab runtime or build wheels manually."
            )
        if not AUTO_PIN_TORCH_FOR_MAMBA:
            raise RuntimeError(
                f"Current torch{torch_major_minor} has no matching Mamba wheels. "
                f"Set AUTO_PIN_TORCH_FOR_MAMBA=True or install torch{target_minor} manually."
            )

        target_version = latest_pypi_torch_version(target_minor)
        pin_manifest = {
            "reason": "Current Colab torch minor has no matching Mamba release wheels.",
            "current_torch_version": torch.__version__,
            "current_torch_minor": torch_major_minor,
            "target_torch_version": target_version,
            "target_torch_minor": target_minor,
            "supported_torch_minors": supported,
            "python_tag": py_tag,
            "cuda_tag": cuda_tag,
            "cxx11abi": abi_tag,
            "uninstalled_torch_companion_packages": (
                torch_companion_packages if UNINSTALL_TORCH_COMPANION_PACKAGES else []
            ),
        }
        wheel_cache_dir.mkdir(parents=True, exist_ok=True)
        pin_path = wheel_cache_dir / "torch_runtime_pin.json"
        pin_path.write_text(json.dumps(pin_manifest, indent=2), encoding="utf-8")
        print(f"Current torch{torch_major_minor} has no compatible Mamba wheels.")
        print(f"Installing torch=={target_version} so Mamba wheels can use torch{target_minor}.")
        print(f"Wrote torch pin manifest: {pin_path}")
        if UNINSTALL_TORCH_COMPANION_PACKAGES:
            print("Removing Torch companion packages that can pin a different torch version:")
            print("  " + ", ".join(torch_companion_packages))
            subprocess.run(
                [sys.executable, "-m", "pip", "uninstall", "-y", *torch_companion_packages],
                check=False,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", f"torch=={target_version}"],
            check=True,
        )
        if RESTART_RUNTIME_AFTER_TORCH_PIN:
            restart_runtime_after_pin()
        raise SystemExit("Restart runtime and rerun the notebook from the top.")

    def select_release_asset(repo: str, role: str) -> dict:
        rel = github_latest_release(repo)
        assets = rel.get("assets", [])
        matches = [a for a in assets if compatible_wheel(a["name"], role)]
        if matches:
            print(f"Selected {role} wheel from {repo} {rel.get('tag_name')}: {matches[0]['name']}")
            return matches[0]

        candidates = [
            a["name"] for a in assets
            if wheel_role(a["name"]) == role and py_tag in a["name"] and "linux_x86_64.whl" in a["name"]
        ]
        print(f"No compatible {role} wheel found in latest release {repo} {rel.get('tag_name')}.")
        print("Closest cp/linux candidates:")
        for name in candidates[:30]:
            print("  -", name)
        raise RuntimeError(
            f"Missing compatible {role} wheel for {py_tag}, torch{torch_major_minor}, "
            f"{cuda_tag}, cxx11abi{abi_tag}. Runtime and wheel tags must match."
        )

    def download_asset(asset: dict, out_dir: Path) -> Path:
        out_dir.mkdir(parents=True, exist_ok=True)
        name = asset["name"]
        out_path = out_dir / name
        if out_path.exists() and out_path.stat().st_size == int(asset.get("size", 0)):
            print(f"Reusing cached download: {out_path}")
            return out_path

        tmp_path = out_path.with_suffix(out_path.suffix + ".partial")
        print(f"Downloading {name}")
        print(f"  -> {out_path}")
        req = urllib.request.Request(asset["browser_download_url"], headers={"User-Agent": "ECG-RAMBA-Colab"})
        with urllib.request.urlopen(req, timeout=900) as resp, tmp_path.open("wb") as f:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
        os.replace(tmp_path, out_path)
        return out_path

    pin_torch_if_needed()

    causal_wheel, mamba_wheel = select_cached_wheels()
    downloaded_assets = []

    if (causal_wheel is None or mamba_wheel is None) and DOWNLOAD_MAMBA_WHEELS_IF_MISSING:
        print("Compatible cached Mamba wheels are missing; downloading release wheels to Drive.")
        if causal_wheel is None:
            asset = select_release_asset("Dao-AILab/causal-conv1d", "causal")
            causal_wheel = download_asset(asset, wheel_cache_dir)
            downloaded_assets.append(asset)
        if mamba_wheel is None:
            asset = select_release_asset("state-spaces/mamba", "mamba")
            mamba_wheel = download_asset(asset, wheel_cache_dir)
            downloaded_assets.append(asset)

    if causal_wheel is None or mamba_wheel is None:
        raise FileNotFoundError(
            "Missing compatible causal_conv1d*.whl and/or mamba_ssm*.whl. "
            "Set DOWNLOAD_MAMBA_WHEELS_IF_MISSING=True or place matching wheels in the Drive cache."
        )

    manifest = {
        "python_tag": py_tag,
        "torch_version": torch.__version__,
        "torch_major_minor": torch_major_minor,
        "cuda_version": cuda_version,
        "cuda_tag": cuda_tag,
        "cxx11abi": abi_tag,
        "wheel_cache_dir": str(wheel_cache_dir),
        "causal_wheel": {
            "path": str(causal_wheel),
            "name": causal_wheel.name,
            "sha256": sha256_file(causal_wheel),
            "bytes": causal_wheel.stat().st_size,
        },
        "mamba_wheel": {
            "path": str(mamba_wheel),
            "name": mamba_wheel.name,
            "sha256": sha256_file(mamba_wheel),
            "bytes": mamba_wheel.stat().st_size,
        },
        "downloaded_assets": [
            {
                "name": a.get("name"),
                "url": a.get("browser_download_url"),
                "size": a.get("size"),
                "digest": a.get("digest"),
            }
            for a in downloaded_assets
        ],
    }
    manifest_path = wheel_cache_dir / "wheel_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Wrote wheel manifest: {manifest_path}")

    print("Installing Mamba wheels from Drive cache")
    print("  causal-conv1d:", causal_wheel.name)
    print("  mamba-ssm    :", mamba_wheel.name)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(causal_wheel), "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(mamba_wheel), "-q"], check=True)

    import mamba_ssm
    print("mamba_ssm import OK:", mamba_ssm.__file__)
else:
    print("Skipping mamba-ssm install. Ensure it is already available before prediction generation.")


## Restore And Verify Stable Drive Artifacts

Restore only files declared by the Drive mirror manifest. Every source and destination is verified by SHA256 before it can be reused.

In [ ]:

stable_mirror = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
if stable_mirror.exists():
    restore_result = run(
        f'python -u scripts/revision/artifact_mirror.py restore '
        f'--mirror-root "{stable_mirror}" --replace-mismatched',
        check=False,
        log_path='reports/revision/logs/oof_mirror_restore.log',
    )
    if restore_result.returncode:
        print('Verified restore unavailable. OOF validation will decide whether inference is required.')
else:
    print('Stable mirror does not exist yet:', stable_mirror)


## Prediction Contract

Every downstream metric script expects NPZ files with `y_true` and `y_prob`, both shaped `(N, C)`. Store prediction files under `reports/revision/predictions/`.

In [ ]:
from pathlib import Path
import numpy as np

pred_dir = Path('reports/revision/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(pred_dir.glob('*.npz')):
    data = np.load(path, allow_pickle=True)
    keys = sorted(data.files)
    print(path, keys)
    if {'y_true', 'y_prob'}.issubset(keys):
        print('  y_true:', data['y_true'].shape, 'y_prob:', data['y_prob'].shape)


## Repair Legacy OOF Aggregation

This step reuses saved slice probabilities only when all five folds and all valid records are present. Otherwise it leaves artifacts untouched.

In [ ]:
RUN_STANDALONE_LEGACY_REAGGREGATION = False

if RUN_STANDALONE_LEGACY_REAGGREGATION:
    run(
        'python -u scripts/revision/02_reaggregate_oof.py --expected-folds 5 --q 3 --if-possible',
        check=False,
        log_path='reports/revision/logs/oof_reaggregate.log',
    )
else:
    print('Standalone repair skipped; Generate OOF Predictions performs this repair automatically.')


## Generate OOF Predictions

In [ ]:
RUN_OOF_EXPORT = True
FORCE_RERUN_OOF = False
RESUME_FOLD_CACHE = True
FORCE_RERUN_FOLDS = False
REQUIRE_HIGH_RAM = True
MIN_SYSTEM_RAM_GB = 24
BATCH_SIZE = 64
NUM_WORKERS = 0
DEBUG_LIMIT_RECORDS = 0
CANONICAL_OOF_KIND = 'final_ema'
CANONICAL_OOF_STEM = 'oof_final_ema'
CANONICAL_FREEZE_MANIFEST = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')

import os
import shutil
import zipfile

if not chapman_zip.is_file() or not zipfile.is_zipfile(chapman_zip):
    raise FileNotFoundError(f'Valid Chapman ZIP is required before OOF generation: {chapman_zip}')
required_model_files = [model_dir / 'folds.pkl']
for fold in range(1, 6):
    required_model_files.append(model_dir / f'fold{fold}_{CANONICAL_OOF_KIND}.pt')
missing_model_files = [str(path) for path in required_model_files if not path.is_file()]
if missing_model_files:
    raise FileNotFoundError(
        'Canonical OOF requires post-fix explicit EMA checkpoints. '
        'Run the retraining notebook/command first. Missing: ' + '; '.join(missing_model_files)
    )
print('Canonical OOF input preflight: Chapman ZIP, folds.pkl, and five fixed-epoch final_ema checkpoints are available.')

page_size = os.sysconf('SC_PAGE_SIZE')
physical_pages = os.sysconf('SC_PHYS_PAGES')
system_ram_gb = page_size * physical_pages / (1024 ** 3)
disk = shutil.disk_usage('/content')
print(f'System RAM : {system_ram_gb:.1f} GiB')
print(f'Local disk : {disk.free / (1024 ** 3):.1f} GiB free')

try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    gpu_ram_gb = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        if torch.cuda.is_available() else 0.0
    )
except Exception:
    gpu_name, gpu_ram_gb = 'unknown', 0.0
print(f'GPU        : {gpu_name} ({gpu_ram_gb:.1f} GiB)')

freeze_command = (
    'python -u scripts/revision/06_freeze_oof.py '
    f'--record-file reports/revision/predictions/{CANONICAL_OOF_STEM}_predictions.npz '
    f'--slice-file reports/revision/predictions/{CANONICAL_OOF_STEM}_slice_predictions.npz '
    f'--summary-file reports/revision/metrics/{CANONICAL_OOF_STEM}_prediction_summary.json '
    f'--class-table reports/revision/tables/{CANONICAL_OOF_STEM}_class_summary.csv '
    f'--run-manifest reports/revision/manifests/{CANONICAL_OOF_STEM}_prediction_run_manifest.json '
    f'--freeze-manifest {CANONICAL_FREEZE_MANIFEST} '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema'
)


def freeze_oof(label):
    print(f'Validating canonical final_ema OOF contract ({label})...')
    result = run(
        freeze_command,
        check=False,
        log_path='reports/revision/logs/oof_final_ema_freeze_validation.log',
    )
    if result.returncode == 0:
        print('final_ema OOF passed checksum, five-fold, 44,186-record, Q=3, and checkpoint validation.')
        return True
    print('final_ema OOF freeze contract is not ready yet; inference may be required.')
    return False


oof_ready = False if FORCE_RERUN_OOF else freeze_oof('existing artifacts')

command = (
    'python -u scripts/revision/01_generate_predictions.py '
    f'--dataset oof --checkpoint-kind {CANONICAL_OOF_KIND} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}'
)
if not RESUME_FOLD_CACHE:
    command += ' --no-resume-fold-cache'
if FORCE_RERUN_FOLDS:
    command += ' --force-rerun-folds'
if DEBUG_LIMIT_RECORDS:
    command += f' --limit-records {DEBUG_LIMIT_RECORDS}'

if RUN_OOF_EXPORT and (FORCE_RERUN_OOF or not oof_ready):
    if REQUIRE_HIGH_RAM and system_ram_gb < MIN_SYSTEM_RAM_GB:
        raise RuntimeError(
            f'Insufficient system RAM: {system_ram_gb:.1f} GiB available, '
            f'but the full final_ema OOF pipeline requires at least {MIN_SYSTEM_RAM_GB} GiB. '
            'Exit code 137 on a standard T4 runtime is a host-RAM kill, not a CUDA batch-size issue. '
            'Select a High-RAM runtime/A100, then rerun from the setup cell. '
            'Do not reduce numerical precision for manuscript predictions.'
        )
    run(command, log_path='reports/revision/logs/oof_final_ema_generate_predictions.log')
    run(freeze_command, log_path='reports/revision/logs/oof_final_ema_freeze_validation.log')
elif oof_ready and not FORCE_RERUN_OOF:
    print('Skipping final_ema OOF inference because the frozen artifact contract passed.')
else:
    print(f'OOF export disabled. Set RUN_OOF_EXPORT=True to execute: {command}')


## Verify OOF Prediction Outputs

In [ ]:
run(
    'python -u scripts/revision/06_freeze_oof.py '
    '--record-file reports/revision/predictions/oof_final_ema_predictions.npz '
    '--slice-file reports/revision/predictions/oof_final_ema_slice_predictions.npz '
    '--summary-file reports/revision/metrics/oof_final_ema_prediction_summary.json '
    '--class-table reports/revision/tables/oof_final_ema_class_summary.csv '
    '--run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --check-only',
    log_path='reports/revision/logs/oof_final_ema_verify_contract.log',
)

expected = [
    Path('reports/revision/predictions/oof_final_ema_predictions.npz'),
    Path('reports/revision/predictions/oof_final_ema_slice_predictions.npz'),
    Path('reports/revision/metrics/oof_final_ema_prediction_summary.json'),
    Path('reports/revision/tables/oof_final_ema_class_summary.csv'),
    Path('reports/revision/manifests/oof_final_ema_prediction_run_manifest.json'),
    Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json'),
]

for path in expected:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

fold_cache_dir = Path('reports/revision/predictions/folds')
fold_cache_files = sorted(fold_cache_dir.glob('oof_fold*_final_ema_*.npz')) if fold_cache_dir.exists() else []
print('\nFinal EMA fold cache files:', len(fold_cache_files))
for path in fold_cache_files:
    print(' -', path, path.stat().st_size, 'bytes')

pred_path = expected[0]
if pred_path.exists():
    data = np.load(pred_path, allow_pickle=True)
    print('keys:', sorted(data.files))
    print('y_true:', data['y_true'].shape)
    print('y_prob:', data['y_prob'].shape)
    print('classes:', list(data['class_names'])[:5], '...', len(data['class_names']))
    for key in [
        'dataset',
        'protocol',
        'config_hash',
        'git_commit',
        'checkpoint_kind',
        'batch_size',
        'aggregation_method',
        'aggregation_q',
        'aggregation_implementation',
        'cache_schema_version',
        'source_config_hash',
        'evaluation_config_hash',
    ]:
        if key in data.files:
            value = data[key]
            print(f'{key}:', value.item() if value.ndim == 0 else value)

summary_path = Path('reports/revision/metrics/oof_final_ema_prediction_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nSummary keys:', sorted(summary.keys()))
    print('metrics:', summary.get('metrics'))
    print('slice_count_summary:', summary.get('slice_count_summary'))

manifest_path = Path('reports/revision/manifests/oof_final_ema_prediction_run_manifest.json')
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('\nManifest runtime:', manifest.get('runtime', {}).get('torch'))
    print('Manifest outputs:')
    for name, info in manifest.get('outputs', {}).items():
        print(' -', name, info.get('path'), info.get('size_bytes'), info.get('sha256', '')[:12])

freeze_path = Path('reports/revision/manifests/oof_final_ema_freeze_manifest.json')
if freeze_path.exists():
    freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
    print('\nFreeze status:', freeze.get('status'))
    print('Manuscript ready:', freeze.get('manuscript_ready'))
    print('Validated records:', freeze.get('validated_records'))
    print('Checkpoint kind:', freeze.get('checkpoint_kind'))
    print('Fold counts:', freeze.get('fold_counts'))
    print('Checkpoint match:', freeze.get('checkpoint_fingerprints_match'))
    if freeze.get('checkpoint_kind') != 'final_ema':
        raise RuntimeError('Canonical freeze manifest must report checkpoint_kind=final_ema')


## Diagnostic Final Checkpoint OOF Audit

This optional diagnostic run evaluates `fold*_final_raw.pt` separately from the manuscript `final_ema` OOF artifacts. It writes `oof_final_raw_*` artifacts and must not replace frozen `oof_final_ema_*` files.


In [ ]:
RUN_FINAL_OOF_AUDIT = False
FORCE_RERUN_FINAL_OOF = False
FINAL_BATCH_SIZE = BATCH_SIZE
FINAL_NUM_WORKERS = NUM_WORKERS
FINAL_N_BOOT = 1000

final_expected = {
    'prediction': Path('reports/revision/predictions/oof_final_raw_predictions.npz'),
    'slice_prediction': Path('reports/revision/predictions/oof_final_raw_slice_predictions.npz'),
    'summary': Path('reports/revision/metrics/oof_final_raw_prediction_summary.json'),
    'class_summary': Path('reports/revision/tables/oof_final_raw_class_summary.csv'),
    'run_manifest': Path('reports/revision/manifests/oof_final_raw_prediction_run_manifest.json'),
    'calibration_ci': Path('reports/revision/metrics/calibration_ci_oof_final_raw_predictions.json'),
}

final_command = (
    'python -u scripts/revision/01_generate_predictions.py '
    f'--dataset oof --checkpoint-kind final_raw --batch-size {FINAL_BATCH_SIZE} --num-workers {FINAL_NUM_WORKERS}'
)
if not RESUME_FOLD_CACHE:
    final_command += ' --no-resume-fold-cache'
if FORCE_RERUN_FOLDS:
    final_command += ' --force-rerun-folds'
if DEBUG_LIMIT_RECORDS:
    final_command += f' --limit-records {DEBUG_LIMIT_RECORDS}'

if not RUN_FINAL_OOF_AUDIT:
    print('Final raw checkpoint audit is disabled. It is diagnostic only and must not replace final_ema OOF.')
    print('Planned command:', final_command)
else:
    final_model_files = [model_dir / f'fold{fold}_final_raw.pt' for fold in range(1, 6)]
    missing_final_models = [str(path) for path in final_model_files if not path.is_file()]
    if missing_final_models:
        raise FileNotFoundError('Final raw audit requires all fold*_final_raw.pt files: ' + '; '.join(missing_final_models))

    final_prediction_ready = all(final_expected[key].exists() for key in ['prediction', 'summary', 'class_summary', 'run_manifest'])
    if FORCE_RERUN_FINAL_OOF or not final_prediction_ready:
        if REQUIRE_HIGH_RAM and system_ram_gb < MIN_SYSTEM_RAM_GB:
            raise RuntimeError(
                f'Insufficient system RAM: {system_ram_gb:.1f} GiB available, '
                f'but the final raw checkpoint OOF audit requires at least {MIN_SYSTEM_RAM_GB} GiB.'
            )
        run(final_command, log_path='reports/revision/logs/oof_final_raw_generate_predictions.log')
    else:
        print('Skipping final raw checkpoint OOF inference because oof_final_raw artifacts already exist.')

    pred_path = final_expected['prediction']
    if not pred_path.exists():
        raise FileNotFoundError(f'Final raw OOF prediction file was not created: {pred_path}')

    with np.load(pred_path, allow_pickle=True) as data:
        print('Final raw prediction keys:', sorted(data.files))
        print('Final raw y_true:', data['y_true'].shape)
        print('Final raw y_prob:', data['y_prob'].shape)
        print('Final raw protocol:', data['protocol'].item() if data['protocol'].ndim == 0 else data['protocol'])
        checkpoint_kind = data['checkpoint_kind'].item() if data['checkpoint_kind'].ndim == 0 else str(data['checkpoint_kind'])
        print('Final raw checkpoint_kind:', checkpoint_kind)
        if tuple(data['y_prob'].shape) != (44186, 27):
            raise RuntimeError(f'Unexpected final raw y_prob shape: {data["y_prob"].shape}')
        if tuple(data['y_true'].shape) != (44186, 27):
            raise RuntimeError(f'Unexpected final raw y_true shape: {data["y_true"].shape}')
        if checkpoint_kind != 'final_raw':
            raise RuntimeError(f'Final raw audit loaded the wrong checkpoint kind: {checkpoint_kind}')
        fold_id = data['fold_id']
        if np.any(fold_id < 0):
            raise RuntimeError(f'Final raw OOF has missing predictions: {int(np.sum(fold_id < 0))}')
        fold_counts = {int(fold): int(np.sum(fold_id == fold)) for fold in sorted(np.unique(fold_id))}
        print('Final raw fold counts:', fold_counts)
        if set(fold_counts) != {1, 2, 3, 4, 5} or sum(fold_counts.values()) != 44186:
            raise RuntimeError(f'Unexpected final raw fold coverage: {fold_counts}')

    calibration_cmd = (
        'python -u scripts/revision/04_calibration_ci.py '
        '--predictions "reports/revision/predictions/oof_final_raw_predictions.npz" '
        '--out "reports/revision/metrics/calibration_ci_oof_final_raw_predictions.json" '
        f'--threshold 0.5 --n-bins 15 --n-boot {FINAL_N_BOOT}'
    )
    if FORCE_RERUN_FINAL_OOF or not final_expected['calibration_ci'].exists():
        run(calibration_cmd, log_path='reports/revision/logs/oof_final_raw_calibration_ci.log')
    else:
        print('Skipping final raw calibration/CI because diagnostic output already exists.')

    for name, path in final_expected.items():
        print(f'{name:16s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

    canonical_calibration = Path('reports/revision/metrics/calibration_ci_oof_final_ema_predictions.json')
    final_calibration = final_expected['calibration_ci']
    if canonical_calibration.exists() and final_calibration.exists():
        best_payload = json.loads(canonical_calibration.read_text(encoding='utf-8'))
        final_payload = json.loads(final_calibration.read_text(encoding='utf-8'))
        rows = []
        for label, payload in [('final_ema/canonical', best_payload), ('final_raw/diagnostic', final_payload)]:
            metrics = payload.get('metrics', {})
            calibration = payload.get('calibration', {})
            rows.append({
                'run': label,
                'f1_macro': metrics.get('f1_macro'),
                'pr_auc_macro': metrics.get('pr_auc_macro'),
                'roc_auc_macro': metrics.get('roc_auc_macro'),
                'ece_macro': calibration.get('ece_macro'),
                'brier_macro': calibration.get('brier_macro'),
            })
        print('\nCanonical final_ema vs final_raw diagnostic comparison:')
        for row in rows:
            print(row)


## Experimental External Prediction Commands

Stable manuscript plan: use Chapman frozen OOF as the main evidence, keep PTB-XL as the only protocol-gated mapped-task external evidence, and leave Georgia/CPSC2021 deferred unless their dataset-specific gates are fixed and rerun. External exports remain isolated with `manuscript_ready=false` until a gate passes.


In [ ]:
# Stable manuscript default: do not rerun optional external exporters.
# Reuse the existing PTB-XL protocol-gated artifact; Georgia/CPSC2021 stay deferred
# unless their mapping/annotation gates are deliberately fixed and rerun.
BUILD_FOLD_PCA = False
RUN_PTBXL_EXPORT = False
RUN_GEORGIA_EXPORT = False
RUN_CPSC2021_EXPORT = False
EXTERNAL_BATCH_SIZE = 128
EXTERNAL_LIMIT_RECORDS = 0
ALLOW_EXTERNAL_EXPORT_FAILURES = True

def first_existing_drive_file(names):
    candidates = [DRIVE_ROOT / name for name in names]
    found = next((path for path in candidates if path.is_file()), None)
    return found, candidates

external_archive_specs = {
    'ptbxl': ['PTB-XL.zip', 'ptb-xl.zip', 'ptbxl.zip'],
    'georgia': ['Georgia.zip'],
    'cpsc2021': ['cpsc2021.zip', 'CPSC2021.zip'],
}
external_archives = {}
for dataset, names in external_archive_specs.items():
    found, candidates = first_existing_drive_file(names)
    external_archives[dataset] = found
    print(f'{dataset} archive:', found if found else 'MISSING', '| checked=', [str(p) for p in candidates])

if BUILD_FOLD_PCA:
    run(
        'python -u scripts/revision/08_build_fold_pca.py --checkpoint-kind final_ema',
        log_path='reports/revision/logs/build_fold_pca.log',
    )
else:
    print('Fold PCA build disabled. External exporter will refuse to run without a complete PCA manifest.')

external_jobs = [
    ('ptbxl', RUN_PTBXL_EXPORT),
    ('georgia', RUN_GEORGIA_EXPORT),
    ('cpsc2021', RUN_CPSC2021_EXPORT),
]
external_export_failures = []
for dataset, enabled in external_jobs:
    if enabled and external_archives.get(dataset) is None:
        message = f'{dataset} export enabled but archive is missing under {DRIVE_ROOT}'
        external_export_failures.append({'dataset': dataset, 'reason': message, 'log_path': None})
        if not ALLOW_EXTERNAL_EXPORT_FAILURES:
            raise FileNotFoundError(message)
        print('External export skipped:', message)
        continue
    command = (
        'python -u scripts/revision/03_generate_external_predictions.py '
        f'--dataset {dataset} --checkpoint-kind final_ema --batch-size {EXTERNAL_BATCH_SIZE} '
        '--allow-experimental'
    )
    if EXTERNAL_LIMIT_RECORDS:
        command += f' --limit-records {EXTERNAL_LIMIT_RECORDS}'
    if enabled:
        log_path = f'reports/revision/logs/{dataset}_generate_predictions.log'
        try:
            run(command, log_path=log_path)
        except subprocess.CalledProcessError as exc:
            external_export_failures.append(
                {
                    'dataset': dataset,
                    'reason': f'export command failed with exit code {exc.returncode}',
                    'log_path': log_path,
                }
            )
            print(f'External export failed for {dataset}; log={log_path}')
            print('This dataset will remain blocked/deferred unless a later successful export creates gate-ready artifacts.')
            if not ALLOW_EXTERNAL_EXPORT_FAILURES:
                raise
    else:
        print(f'{dataset}: disabled; set RUN_{dataset.upper()}_EXPORT=True to run')

if external_export_failures:
    print('External export failures recorded:')
    for failure in external_export_failures:
        print(f"- {failure['dataset']}: {failure['reason']} | log={failure['log_path']}")
    print('Continue to the protocol gate with EXTERNAL_GATE_STRICT=False so failures are recorded as blocked/deferred.')

print('Experimental outputs are written under reports/revision/experimental.')


## External Protocol Gate

Run this after PTB-XL experimental export artifacts exist. For the stable manuscript package, gate PTB-XL only. Georgia and CPSC2021 should remain deferred until their label-mapping/window-annotation protocols produce gate-ready artifacts. Passing the gate permits mapped-task external reporting only, not unqualified external-transfer advantage.


In [ ]:
# Stable manuscript default: gate PTB-XL only. Notebook 07 will mark
# Georgia/CPSC2021 as deferred when their gate artifacts are absent.
RUN_EXTERNAL_PROTOCOL_GATE = False
EXTERNAL_GATE_DATASETS = 'ptbxl'
EXTERNAL_GATE_N_BOOT = 1000
EXTERNAL_GATE_STRICT = False
EXTERNAL_GATE_REUSE_EXISTING = True

external_gate_command = (
    'python -u scripts/revision/18_external_protocol_gate.py '
    f'--dataset all --expected-checkpoint-kind final_ema '
    f'--threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_GATE_N_BOOT}'
)
if EXTERNAL_GATE_STRICT:
    external_gate_command += ' --strict'
if EXTERNAL_GATE_REUSE_EXISTING:
    external_gate_command += ' --reuse-existing'
if EXTERNAL_GATE_DATASETS.strip() and EXTERNAL_GATE_DATASETS.strip().lower() != 'all':
    parts = [item.strip() for item in EXTERNAL_GATE_DATASETS.split(',') if item.strip()]
    external_gate_command = (
        'python -u scripts/revision/18_external_protocol_gate.py '
        + ' '.join(f'--dataset {item}' for item in parts)
        + f' --expected-checkpoint-kind final_ema --threshold 0.5 --n-bins 15 --n-boot {EXTERNAL_GATE_N_BOOT}'
    )
    if EXTERNAL_GATE_STRICT:
        external_gate_command += ' --strict'
    if EXTERNAL_GATE_REUSE_EXISTING:
        external_gate_command += ' --reuse-existing'

if RUN_EXTERNAL_PROTOCOL_GATE:
    run(
        external_gate_command,
        log_path='reports/revision/logs/external_protocol_gate.log',
    )
else:
    print('External protocol gate disabled. For the stable manuscript run, set RUN_EXTERNAL_PROTOCOL_GATE=True with EXTERNAL_GATE_DATASETS=\'ptbxl\'.')
    print('Georgia/CPSC2021 are intentionally deferred unless their dataset-specific gates are repaired and rerun.')
    print('Planned command:', external_gate_command)

for dataset in ['ptbxl', 'georgia', 'cpsc2021']:
    for path in [
        Path(f'reports/revision/metrics/external_{dataset}_protocol_gate.json'),
        Path(f'reports/revision/tables/table_external_{dataset}_label_mapping.csv'),
        Path(f'reports/revision/tables/table_external_{dataset}_metrics.csv'),
        Path(f'reports/revision/manifests/external_{dataset}_protocol_gate_manifest.json'),
    ]:
        print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
summary = Path('reports/revision/metrics/external_protocol_gate_summary.csv')
print(f'{summary}: exists={summary.exists()} size={summary.stat().st_size if summary.exists() else None}')
if summary.exists():
    import pandas as pd
    gate_summary = pd.read_csv(summary)
    display(gate_summary[['dataset', 'status', 'protocol_gate_passed', 'manuscript_ready', 'issue_count', 'reused_existing', 'gate_cache_key']])


## Re-run Inventory

In [ ]:
!python scripts/revision/05_artifact_inventory.py


## Mirror Artifacts To Stable Drive Cache

In [ ]:
run(
    'python -u scripts/revision/06_freeze_oof.py '
    '--record-file reports/revision/predictions/oof_final_ema_predictions.npz '
    '--slice-file reports/revision/predictions/oof_final_ema_slice_predictions.npz '
    '--summary-file reports/revision/metrics/oof_final_ema_prediction_summary.json '
    '--class-table reports/revision/tables/oof_final_ema_class_summary.csv '
    '--run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-records 44186 --expected-folds 5 --q 3 --expected-checkpoint-kind final_ema --check-only',
    log_path='reports/revision/logs/oof_final_ema_pre_publish_contract.log',
)

source_root = Path('reports/revision')
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/oof_mirror_publish.log',
)
